# Phase 1 independent member reproduction — Real-Absher

Another team member executes this fixed experiment in their own fresh Kaggle GPU runtime and records their actual name. The main run was executed by Fahad. A repeat by the same person is useful, but does not fulfil the instructor's another-member requirement.

Attach the corrections **code dataset** and the original **data-dfd image dataset**. Use a T4 GPU and enable Internet. This notebook trains all four strategies anew; it uses the common public pretrained initialization for transfer learning. It does not restore the main run's trained checkpoints.

Run the cells below in order. Keep the session open while the experiment runs. The experiment itself runs in a fresh Python process so notebook-memory package versions do not affect it. All model choices use validation cross-entropy; the already reported test cohort is repeated only to reproduce this fixed protocol.


In [ ]:
from pathlib import Path
import sys, subprocess

# The member actually executing this notebook enters their own full name.
OPERATOR_NAME = "ENTER YOUR FULL NAME"
assert OPERATOR_NAME != "ENTER YOUR FULL NAME" and OPERATOR_NAME.strip(), "Enter the actual operator's full name."
assert OPERATOR_NAME.strip().casefold() != "Fahad Falah H. AlQahtani".casefold(), "This independent member run must be executed by another team member."
projects = sorted({p.parent for p in Path('/kaggle/input').rglob('phase1_training.py')})
assert len(projects) == 1, f'Attach one corrections code dataset: {projects}'
PROJECT = projects[0]
print('Code dataset:', PROJECT)
print('Actual operator:', OPERATOR_NAME)


In [ ]:
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    '-r', str(PROJECT / 'requirements.txt'),
    'torchaudio==2.11.0+cu128'
])
print('Installation complete. The next cell starts a fresh Python process.')


## Execute the fixed experiment

The next cell prepares the original data and the reserved holdout, checks the immutable development split and protocol digest, trains scratch-unweighted, scratch-weighted, frozen and full, locks the validation-selected model, evaluates the predefined validation interventions and final test, and exports the outputs.

Preserve the complete output if the job fails; send its traceback for recovery. A completed run prints `Independent execution complete` and two ZIP filenames. This notebook has been checked for structure and Python syntax; its actual independent execution is still pending.


In [ ]:
job_code = r'''PRIMARY_OPERATOR = 'Fahad Falah H. AlQahtani'
CORE_HASHES = {'phase1_training.py': '118fbab1389716e3a70442a1302fec6324391f0f804c3185a638a10da3c7a517', 'phase1_corrections.py': 'abecc3bc91506639640d3bc4a89d09b9ef0a0b9e7a57b13cbef8ec67fce5edd5', 'dani_provenance.py': '7e385f1fc5d4a507c7ae85e9fd478cdee8046ae0e6a5d232a479bfa1f5a090ad'}
from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED
from importlib.metadata import version
import sys, json, shutil, subprocess, hashlib
import pandas as pd
import torch, torchvision, torchaudio, transformers

PROJECT = Path(sys.argv[1]).resolve()
OPERATOR_NAME = sys.argv[2].strip()
sys.path.insert(0, str(PROJECT))
RUN = Path('/kaggle/working/phase1_independent_run')
assert not RUN.exists() or not any(RUN.iterdir()), 'Use a fresh runtime; this output folder already contains a run.'
assert OPERATOR_NAME and OPERATOR_NAME.casefold() != PRIMARY_OPERATOR.casefold(), 'Another member must enter their own name and execute this run.'
for filename, expected in CORE_HASHES.items():
    with (PROJECT / filename).open('rb') as stream:
        assert hashlib.file_digest(stream, 'sha256').hexdigest() == expected, 'Training source differs: ' + filename
for module, wanted in [(torch,'2.11.0+cu128'), (torchvision,'0.26.0+cu128'), (torchaudio,'2.11.0+cu128'), (transformers,'5.16.1')]:
    assert module.__version__ == wanted, module.__name__ + ': unexpected version'
assert torch.cuda.is_available(), 'Enable the T4 GPU.'
print('Independent operator:', OPERATOR_NAME, flush=True)
print('GPU:', torch.cuda.get_device_name(0), flush=True)
print('Preparing the fixed data protocol', flush=True)

from zipfile import ZipFile
DATA_ROOT=Path('/kaggle/working/phase1_images')
PREPARED=Path('/kaggle/working/phase1_prepared')
RUN=Path('/kaggle/working/phase1_independent_run')
CACHE=Path('/kaggle/working/dani_publisher_cache')
DATA_ROOT.mkdir(parents=True,exist_ok=True);PREPARED.mkdir(parents=True,exist_ok=True)
if not (DATA_ROOT/'data/manifest.csv').exists():
    manifests=[x for x in Path('/kaggle/input').rglob('manifest.csv') if x.parent.name=='data']
    if len(manifests)==1: shutil.copytree(manifests[0].parent,DATA_ROOT/'data',dirs_exist_ok=True)
    else:
        archives=list(Path('/kaggle/input').rglob('data.zip'))
        assert len(archives)==1, 'Attach the prepared DANI data; set the archive path explicitly if there are several.'
        with ZipFile(archives[0]) as archive:
            for info in archive.infolist():
                if info.is_dir() or not info.filename.startswith('data/') or '/._' in info.filename: continue
                target=(DATA_ROOT/info.filename).resolve()
                assert target.is_relative_to(DATA_ROOT.resolve()), 'Unsafe archive path'
                target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(archive.read(info))
assert (DATA_ROOT/'data/manifest.csv').is_file()
EVIDENCE=PROJECT/'docs/phase1_corrections/evidence'
print('Data root:',DATA_ROOT)

import pandas as pd
from phase1_corrections import inspect_images,attach_links,development_split,audit_report,fingerprint
clean,counts=inspect_images(DATA_ROOT/'data/manifest.csv',DATA_ROOT,PREPARED/'cleaning')
links=pd.read_csv(EVIDENCE/'dani_source_links.csv')
all_old=attach_links(pd.read_csv(PREPARED/'cleaning/all_readable_manifest.csv',dtype={'phash':str}),links)
all_old.to_csv(PREPARED/'all_historical_linked_manifest.csv',index=False)
import hashlib

# Load the fixed split shipped with the corrections dataset.
split_file = EVIDENCE / "corrected_development_manifest.csv"
assert hashlib.sha256(split_file.read_bytes()).hexdigest() == "15f17c6e5d7f6c5df0ce4ac0a6f09023b797e6710e3dc64870bfb3ecbf0ad138", "Fixed split file changed"
frozen = pd.read_csv(split_file, dtype={"sha256": str, "phash": str})

# Verify the fixed records against the images checked in this run.
assert frozen.relative_path.is_unique and frozen.sha256.is_unique
assert set(frozen.sha256) == set(clean.sha256), "Retained image population changed"
observed = all_old.set_index("relative_path").reindex(frozen.relative_path)
for field in ["sha256", "label", "group_id"]:
    assert observed[field].tolist() == frozen[field].tolist(), f"Image identity mismatch: {field}"

# Save this exact split for training and holdout preparation.
development = frozen.copy()
assert set(development.split) == {"train", "val"}
assert development.groupby("group_id").split.nunique().max() == 1
assert development.group_id.nunique() == 498
development.to_csv(PREPARED / "development_manifest.csv", index=False)

print(counts)
print(development.groupby('split').agg(images=('label','size'),groups=('group_id','nunique')))
print(pd.crosstab(development.split,development.label))
assert development.groupby('group_id').split.nunique().max()==1
assert development.group_id.nunique()==498

fixed=pd.read_csv(EVIDENCE/'label_audit_fixed_sample.csv')
decisions=pd.read_csv(EVIDENCE/'label_audit.csv')
for row in fixed.itertuples():
    assert fingerprint(DATA_ROOT/row.relative_path)[0]==row.sha256, 'Audit image identity changed'
report=audit_report(fixed,decisions)
assert report['complete'], 'Complete the fixed audit before training'
print(report)
decisions.to_csv(PREPARED/'label_audit.csv',index=False)
(PREPARED/'label_audit_report.json').write_text(json.dumps(report,indent=2))

from dani_provenance import acquire_holdout
protocol=acquire_holdout(EVIDENCE/'reserved_holdout_sources.csv',PREPARED/'development_manifest.csv',
    PREPARED/'all_historical_linked_manifest.csv',DATA_ROOT,CACHE,PREPARED)
PROTOCOL=PREPARED/'protocol_manifest.csv'
print(protocol.groupby(['split','label']).size())
print(json.loads((PREPARED/'holdout_verification.json').read_text()))
assert protocol.groupby('group_id').split.nunique().max()==1

MAIN_CONFIG = {'seed': 42, 'image_size': 224, 'batch_size': 8, 'accumulation_steps': 1, 'cnn_epochs': 25, 'transfer_epochs': 15, 'cnn_lr': 0.01, 'head_lr': 0.01, 'backbone_lr': 0.001, 'momentum': 0.9, 'weight_decay': 0.0001, 'patience': 5, 'min_delta': 0.0001, 'threshold': 0.5}
MAIN_RUN_ID = '347c0f0188f04f8c873e2eac415e5723'

from phase1_training import train_and_lock, validation_robustness, final_test, digest, DEFAULTS
from phase1_corrections import analyse_historical
assert digest(PROTOCOL) == '0544ea4ee4940c56874855021b0f0a5873f121ddeaf26c4880894374529eeb8b', 'Protocol differs from the main run'
assert DEFAULTS == MAIN_CONFIG, 'Hyperparameters differ from the main run'
print('Training all four strategies from the prescribed initializations', flush=True)
comparison = pd.DataFrame(train_and_lock(PROTOCOL, DATA_ROOT, RUN))
environment = json.loads((RUN / 'environment.json').read_text())
assert environment['run_id'] != MAIN_RUN_ID
assert environment['code_hashes'] == CORE_HASHES
environment['operator'] = OPERATOR_NAME
environment['reproduction_of_run_id'] = MAIN_RUN_ID
environment['execution_note'] = 'Operator reports execution of this fixed experiment in a fresh runtime; software metadata alone does not authenticate human independence.'
(RUN / 'environment.json').write_text(json.dumps(environment, indent=2))
assert digest(RUN / 'config.json') == 'c4570d4f3fe07c2a37bf888047b6216854dfb5f0bef8712f85a1f7a30a45acc2'
print(comparison.to_string(index=False), flush=True)
print((RUN / 'selection_locked.json').read_text(), flush=True)

print('Measuring the predefined validation interventions', flush=True)
robustness = validation_robustness(PROTOCOL, DATA_ROOT, RUN)
print(robustness.to_string(index=False), flush=True)
print('Evaluating the fixed final test after validation-only selection', flush=True)
metrics = final_test(PROTOCOL, DATA_ROOT, RUN)
print(metrics.to_string(index=False), flush=True)
record = {'process_completed_at_utc':datetime.now(timezone.utc).isoformat(), 'python':sys.version,
          'operator':OPERATOR_NAME,'torch':torch.__version__,'torchvision':torchvision.__version__,
          'torchaudio':torchaudio.__version__,'transformers':transformers.__version__,
          'gpu':torch.cuda.get_device_name(0)}
(RUN / 'evaluation_environment.json').write_text(json.dumps(record, indent=2))
analysis, slices, tests = analyse_historical(RUN, DATA_ROOT, RUN / 'error_analysis')
for path in (RUN / 'error_analysis').glob('*_error_review.csv'):
    review = pd.read_csv(path)
    review['cause_status'] = 'Descriptive association only; controlled validation interventions are recorded separately'
    review.to_csv(path, index=False)
print(slices.loc[slices.dimension.isin(['scene','source_model'])].to_string(index=False), flush=True)
models = Path('/kaggle/working/Phase1_Independent_Models.zip')
subprocess.check_call([sys.executable,str(PROJECT / 'scripts/export_bundle.py'),'--run',str(RUN),'--output',str(models)])
results = Path('/kaggle/working/Phase1_Independent_Results_Only.zip')
with ZipFile(results,'w',ZIP_DEFLATED) as archive:
    for path in sorted(RUN.rglob('*')):
        relative = path.relative_to(RUN)
        if path.is_file() and 'wandb' not in relative.parts and path.suffix in {'.csv','.json','.txt','.sha256'}:
            archive.write(path, relative.as_posix())
print('Independent execution complete. Download:', models, flush=True)
print('Return this smaller file for comparison:', results, flush=True)
'''

subprocess.check_call([sys.executable, '-u', '-c', job_code, str(PROJECT), OPERATOR_NAME])


## Save and return the real outputs

Download both `Phase1_Independent_Models.zip` and `Phase1_Independent_Results_Only.zip` from Kaggle's output files. Keep the models ZIP. Return the smaller results ZIP to Fahad for comparison with the original results. Download the executed notebook too.

After both actual results archives are available, extract them into distinct folders and run the provided comparison tool, replacing the member text with the exact operator name:

```bash
python scripts/compare_reproduction.py --original original_run --rerun teammate_run --member "EXACT RECORDED OPERATOR NAME" --independent-run-confirmed --output reproduction_record
```

The comparison checks distinct run IDs and operators, identical training source, configuration and protocol, and reports accuracy, macro-F1 and ECE side by side. Small numerical differences should be reported, not replaced with copied numbers. Software records support the member's attestation but cannot independently authenticate who operated the account.
